In [5]:
import pandas as pd
import os

# 1. Load the EA Sports Male Players data (ignoring the female files)
# Update the path below based on the exact name of the csv inside your folder
ea_male_players_path = '../data/raw/Kaggle_EA_Sports_Player_Dataset/male_players.csv'
df_players = pd.read_csv(ea_male_players_path)

# Let's take a quick look at the data
df_players.head()

C:\Users\87591\AppData\Local\Temp\ipykernel_32988\2316725279.py:7: DtypeWarning: Columns (0: gk) have mixed types. Specify dtype option on import or set low_memory=False.
  df_players = pd.read_csv(ea_male_players_path)


,player_id,player_url,fifa_version,fifa_update,update_as_of,short_name,long_name,player_positions,overall,potential,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
0,231747,/player/231747/kylian-mbappe/240002,24.0,2.0,2023-09-22,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,94,...,63+3,63+3,63+3,68+3,63+3,54+3,54+3,54+3,63+3,18+3
1,239085,/player/239085/erling-haaland/240002,24.0,2.0,2023-09-22,E. Haaland,Erling Braut Haaland,ST,91,94,...,63+3,63+3,63+3,62+3,60+3,62+3,62+3,62+3,60+3,19+3
2,192985,/player/192985/kevin-de-bruyne/240002,24.0,2.0,2023-09-22,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,...,80+3,80+3,80+3,79+3,75+3,70+3,70+3,70+3,75+3,21+3
3,158023,/player/158023/lionel-messi/240002,24.0,2.0,2023-09-22,L. Messi,Lionel Andrés Messi Cuccittini,"CF, CAM",90,90,...,63+3,63+3,63+3,64+3,59+3,49+3,49+3,49+3,59+3,19+3
4,165153,/player/165153/karim-benzema/240002,24.0,2.0,2023-09-22,K. Benzema,Karim Benzema,"CF, ST",90,90,...,64+3,64+3,64+3,64+3,60+3,55+3,55+3,55+3,60+3,18+3


In [6]:
# 2. keep only yhe columns we need to save memory

cols_to_keep = ['fifa_version', 'short_name', 'nationality_name', 'overall']
df_players_subset = df_players[cols_to_keep].copy()

df_players_subset.head()


,fifa_version,short_name,nationality_name,overall
0,24.0,K. Mbappé,France,91
1,24.0,E. Haaland,Norway,91
2,24.0,K. De Bruyne,Belgium,91
3,24.0,L. Messi,Argentina,90
4,24.0,K. Benzema,France,90


In [7]:
# 3. Sort the data so the best players are at the top for each country and year

df_players_subset = df_players_subset.sort_values(by=['fifa_version', 'overall'], ascending=False)
df_players_subset.head()


,fifa_version,short_name,nationality_name,overall
0,24.0,K. Mbappé,France,91
1,24.0,E. Haaland,Norway,91
2,24.0,K. De Bruyne,Belgium,91
3,24.0,L. Messi,Argentina,90
4,24.0,K. Benzema,France,90


In [8]:
# 4. Group by Year and Country, then grab the top 20 players for each

# (20 players represents a solid core of a World Cup squad)

top_20_players = df_players_subset.groupby(['fifa_version', 'nationality_name']).head(20)
top_20_players.head()


,fifa_version,short_name,nationality_name,overall
0,24.0,K. Mbappé,France,91
1,24.0,E. Haaland,Norway,91
2,24.0,K. De Bruyne,Belgium,91
3,24.0,L. Messi,Argentina,90
4,24.0,K. Benzema,France,90


In [9]:
from numpy._typing import _nested_sequence
# 5. Calculate the average 'overall' rating of those 20 players

squad_strength = top_20_players.groupby(['fifa_version', 'nationality_name'])['overall'].mean().reset_index()
squad_strength.head()   

,fifa_version,nationality_name,overall
0,15.0,Albania,68.3
1,15.0,Algeria,73.3
2,15.0,Angola,64.0
3,15.0,Antigua and Barbuda,59.5
4,15.0,Argentina,82.2


In [10]:
# 6. Rename the columns so they are clean and easy to read later

squad_strength.rename(columns={
    'fifa_version': 'year', 
    'nationality_name': 'country',
    'overall': 'ea_squad_score'
}, inplace=True)

In [11]:
# 7. Convert the 'year' from EA's format (e.g., FIFA 24.0 -> 2024)

squad_strength['year'] = squad_strength['year'] + 2000

In [12]:
# View the results!
display(squad_strength.sort_values(by=['year', 'ea_squad_score'], ascending=[False, False]).head(10))

,year,country,ea_squad_score
1490,2024.0,France,85.30
1455,2024.0,Brazil,85.15
1494,2024.0,Germany,84.90
1483,2024.0,England,84.70
1574,2024.0,Spain,84.15
1557,2024.0,Portugal,83.80
1513,2024.0,Italy,83.65
1444,2024.0,Argentina,83.25
1543,2024.0,Netherlands,82.10
1450,2024.0,Belgium,81.25


In [13]:
import os

os.makedirs('../data/processed', exist_ok=True)

# Save the dataframe to a CSV file
squad_strength.to_csv('../data/processed/ea_squad_strength.csv', index=False)
print("EA Squad Strength saved successfully!")

EA Squad Strength saved successfully!


In [14]:
# 1. Load players fresh (Only keeping the 2 columns we need)
players_path = '../data/raw/Kaggle_Football_Data_From_Transfermarket/players.csv'
df_p_fresh = pd.read_csv(players_path)[['player_id', 'country_of_citizenship']]

In [15]:
# 2. Load valuations fresh (Only keeping the 3 columns we need)
val_path = '../data/raw/Kaggle_Football_Data_From_Transfermarket/player_valuations.csv'
# NOTE: If this throws an error, it means the column name in player_valuations is different!
df_v_fresh = pd.read_csv(val_path)[['player_id', 'date', 'market_value_in_eur']]

In [16]:
# 3. Merge them together
df_merged_fresh = pd.merge(df_v_fresh, df_p_fresh, on='player_id', how='inner')

In [17]:
# 4. Extract the year
df_merged_fresh['year'] = pd.to_datetime(df_merged_fresh['date']).dt.year


In [18]:
# 5. Drop empty rows
df_merged_fresh.dropna(subset=['market_value_in_eur', 'country_of_citizenship'], inplace=True)

In [19]:
# 6. Group by Year, Country, Player and get their MAX value for that year
df_max_val = df_merged_fresh.groupby(['year', 'country_of_citizenship', 'player_id'])['market_value_in_eur'].max().reset_index()

In [20]:
# 7. Sort and get top 23 players per country per year
df_sorted = df_max_val.sort_values(by=['year', 'country_of_citizenship', 'market_value_in_eur'], ascending=[True, True, False])
top_23 = df_sorted.groupby(['year', 'country_of_citizenship']).head(23)

In [21]:
# 8. SUM the values to get total squad value
squad_value = top_23.groupby(['year', 'country_of_citizenship'])['market_value_in_eur'].sum().reset_index()

In [22]:
# 9. Clean names
squad_value.rename(columns={
    'country_of_citizenship': 'country',
    'market_value_in_eur': 'tm_squad_value_eur'
}, inplace=True)

In [23]:
# Look at the 2022 results
display(squad_value[squad_value['year'] == 2022].sort_values(by='tm_squad_value_eur', ascending=False).head(10))

,year,country,tm_squad_value_eur
2879,2022,England,1504000000
2887,2022,France,1358000000
2849,2022,Brazil,1282000000
2986,2022,Spain,1162000000
2964,2022,Portugal,1103000000
2891,2022,Germany,985000000
2912,2022,Italy,844000000
2833,2022,Argentina,816000000
2945,2022,Netherlands,799000000
2843,2022,Belgium,700000000


In [24]:
# save the squad value to csv
squad_value.to_csv('../data/processed/tm_squad_value.csv', index=False)

## Next Dataset: FIFA World Rankings

In [ ]:
# 1. Load the dataset and check the first rows of the data
df_fifa_ranking = pd.read_csv("../data/raw/Kaggle_Fifa_World_Ranking/fifa_ranking-2024-06-20.csv")
df_fifa_ranking.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [29]:
# 2. Keep only the columns that matter to our model
cols_to_keep = ['rank', 'country_full', 'total_points', 'confederation', 'rank_date']
df_clean_ranks = df_fifa_ranking[cols_to_keep].copy()


In [ ]:
# 3. Rename columns to match our standard format
df_clean_ranks.rename(columns={
    'country_full': 'country',
    'rank': 'fifa_rank',
    'total_points': 'fifa_points'
}, inplace=True)

In [31]:
# 4. Convert the date string into a proper Python Datetime object
df_clean_ranks['rank_date'] = pd.to_datetime(df_clean_ranks['rank_date'])

In [32]:
# 5. Sort by date and rank just to keep it tidy
df_clean_ranks.sort_values(by=['rank_date', 'fifa_rank'], ascending=[False, True], inplace=True)

In [33]:
# Let's look at the Top 5 teams in the world as of June 2024!
display(df_clean_ranks.head(5))

,fifa_rank,country,fifa_points,confederation,rank_date
67471,1.0,Argentina,1860.14,CONMEBOL,2024-06-20
67334,2.0,France,1837.47,UEFA,2024-06-20
67333,3.0,Belgium,1797.98,UEFA,2024-06-20
67332,4.0,Brazil,1791.85,CONMEBOL,2024-06-20
67331,5.0,England,1787.88,UEFA,2024-06-20


In [34]:
# 6. Save it to our processed folder
df_clean_ranks.to_csv('../data/processed/fifa_rankings.csv', index=False)
print("\nFIFA Rankings cleaned and saved successfully!")


FIFA Rankings cleaned and saved successfully!
